In [1]:
import os
import json
import asyncio
import openai               
from dotenv import load_dotenv 
import pandas as pd  
load_dotenv()

True

In [2]:
category_schemas = json.load(open('../config/sdoh_extraction_schema_updated.json', 'r'))

In [3]:
client = openai.OpenAI()


In [5]:
data = pd.read_csv("../../1.tag/data/mimic3/mimic3_adhf_social_history_0406_tags.csv", index_col=0)

In [6]:
data['tags'] = data['tags'].fillna('')

In [12]:
import json
import pandas as pd
import copy  # Add this to copy your schema safely
from tqdm import tqdm
import concurrent.futures

def process_single_admission(args):
    hadm_id, group_df, context_window_size = args
    doc_sentences = group_df.to_dict(orient='records')
    processed_list = []
    
    # 1. Create a mapping for the tags that have spaces in the schema
    tag_to_schema_map = {
        "SubstanceUse": "Substance Use",
        "MentalHealth": "Mental Health"
    }
    
    for i, row in enumerate(doc_sentences):
        current_sentence = row['social_history_sentences']
        current_tags = row['tags']
        
        item_result = {
            "index": i,
            "sentence": current_sentence,
            "predicted_tags": current_tags,
            "extracted_predictions": {}
        }
        
        if pd.isna(current_tags) or str(current_tags).strip() == "":
            processed_list.append(item_result)
            continue
        
        current_categories = [cat.strip() for cat in str(current_tags).split(",")]
        
        start_index = max(0, i - context_window_size)
        previous_sentences = [doc['social_history_sentences'] for doc in doc_sentences[start_index:i]]
        previous_context = " ".join(previous_sentences)
        
        for category in current_categories:
            # 2. Get the correct key for your category_schemas dictionary
            schema_key = tag_to_schema_map.get(category, category)
            
            if schema_key not in category_schemas:
                continue
                
            # 3. Use deepcopy so we don't permanently change your original schema dictionary
            func_schema = copy.deepcopy(category_schemas[schema_key])
            
            # 4. Remove the space for the OpenAI API requirement
            safe_function_name = func_schema["name"].replace(" ", "")
            func_schema["name"] = safe_function_name
            
            prompt = f"""Context sentences for reference: "{previous_context}"
Current Sentence to process: "{current_sentence}"
Target Category: {schema_key}

Task: Extract structured information strictly for the Current Sentence. Use the Context sentences ONLY to resolve pronouns or identify the missing 'Experiencer'. Do not extract events that only occurred in the Context. Return the JSON required by the {schema_key} schema."""

            try:
                response = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[{"role": "user", "content": prompt}],
                    functions=[func_schema],
                    # 5. Use the safe name without spaces here as well
                    function_call={"name": safe_function_name}
                )

                parsed_result_str = response.choices[0].message.function_call.arguments
                
                try:
                    parsed_result = json.loads(parsed_result_str)
                except json.JSONDecodeError:
                    parsed_result = parsed_result_str
                    
                # Store it under the original RoBERTa category name (e.g., 'SubstanceUse')
                item_result["extracted_predictions"][category] = parsed_result

            except Exception as e:
                item_result["extracted_predictions"][category] = f"API Error: {str(e)}"
        
        processed_list.append(item_result)
        
    return hadm_id, processed_list

In [13]:
def process_all_admissions_parallel(df, context_window_size=3, max_workers=5):
    """
    Distributes the processing of each hadm_id across multiple threads.
    """
    final_dict = {}
    
    # First, prepare the arguments for each thread
    # We create a list of tuples: (hadm_id, dataframe_for_that_id, context_window_size)
    tasks = []
    for hadm_id, group in df.groupby('hadm_id'):
        tasks.append((hadm_id, group, context_window_size))
        
    print(f"Total admissions to process: {len(tasks)}")
    
    # Use ThreadPoolExecutor for I/O bound parallelization
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # executor.map will run the tasks and keep the progress bar working
        # We wrap it in tqdm to track the completed threads
        results = list(tqdm(executor.map(process_single_admission, tasks), total=len(tasks), desc="Processing in Parallel"))
        
    # Reconstruct the final dictionary from the parallel results
    for hadm_id, processed_list in results:
        final_dict[hadm_id] = processed_list
        
    return final_dict

In [14]:
extracted_results = process_all_admissions_parallel(data, context_window_size=2, max_workers=5)

Total admissions to process: 3600


Processing in Parallel: 100%|██████████████| 3600/3600 [39:14<00:00,  1.53it/s]


In [26]:
import math
def absolute_clean(obj):
    """
    Recursively searches and destroys any form of pandas, numpy, or Python nulls.
    """
    if isinstance(obj, dict):
        return {k: absolute_clean(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [absolute_clean(v) for v in obj]
    
    # pd.isna() is much stronger. It catches np.nan, pd.NA, and float('nan')
    try:
        if pd.isna(obj):
            return None  # Translates to strict JSON 'null'
    except Exception:
        pass # In case pd.isna() encounters a weird type it can't check
        
    return obj

# 1. Clean the dictionary currently in your memory
cleaned_results = absolute_clean(extracted_results)

# 2. Save it to your JSON file
output_path = "../output/mimic3/extracted_results.json"

with open(output_path, "w", encoding="utf-8") as f:
    # indent=4 makes the JSON file readable
    json.dump(cleaned_results, f, indent=4)